# Chapter 12 — Menus, message boxes, and file/directory dialogs

**Book role:** Core GUI  
**Suggested book space:** about 8 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Build menus and connect semantic commands.
- Use message boxes for confirmations and important errors.
- Use file dialogs to choose paths without hard-coding locations.
- Handle cancellation and filesystem errors explicitly.

## 1. Menus as discoverability and command organization

A menu bar groups actions in recognizable categories such as File, Edit, View, Data, and Help. Add separators between related command groups and keyboard accelerators that match the actual bindings. Menu entries should call the same application methods as buttons; duplicating save logic in multiple callbacks creates inconsistent behavior.

For analytics software, menu actions often include Open Dataset, Save Filtered Data, Export Report, Refresh, Preferences, and About. Destructive actions such as overwriting a workbook or deleting records should state the target and consequence. Disable actions that are not currently available rather than allowing a menu item to fail ambiguously.

### Worked example 1: Create a menu bar

A menu entry’s command should call a shared application method in a real application. The example uses status updates to make menu routing visible.

In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox

root = tk.Tk()
root.title("Menu example")
root.geometry("500x260")
status = tk.StringVar(value="Ready")
ttk.Label(root, textvariable=status, padding=20).pack(anchor="w")
menu_bar = tk.Menu(root)
file_menu = tk.Menu(menu_bar, tearoff=False)
file_menu.add_command(label="Open…", accelerator="Ctrl+O", command=lambda: status.set("Open selected"))
file_menu.add_command(label="Export…", accelerator="Ctrl+E", command=lambda: status.set("Export selected"))
file_menu.add_separator()
file_menu.add_command(label="Exit", command=root.destroy)
menu_bar.add_cascade(label="File", menu=file_menu)
help_menu = tk.Menu(menu_bar, tearoff=False)
help_menu.add_command(label="About", command=lambda: messagebox.showinfo("About", "Tkinter course demo", parent=root))
menu_bar.add_cascade(label="Help", menu=help_menu)
root.configure(menu=menu_bar)
root.mainloop()

## 2. Dialogs return values and may be cancelled

`filedialog.askopenfilename()` returns a path-like string or an empty string when cancelled. `asksaveasfilename()` chooses a destination but does not save the file for you. `askdirectory()` behaves similarly for folders. Always treat cancellation as a normal branch, not as an exception. Restrict visible file types to relevant formats while still validating the extension and file contents after selection.

Message-box functions return values that should control the next step. A Yes/No confirmation is meaningful only if the code branches on the answer. Avoid stacking modal dialogs or showing technical stack traces to ordinary users. Present a short explanation and provide a log entry for details that help debugging.

### Worked example 2: Open a CSV with cancellation and error handling

`withdraw()` hides the root window but does not remove it. Production code should keep the root and dialog lifecycle inside the application and show friendly messages while recording diagnostic context.

In [ ]:
from pathlib import Path
import tkinter as tk
from tkinter import filedialog, messagebox
import pandas as pd

root = tk.Tk()
root.withdraw()  # Hide the root while asking for a file.
selected = filedialog.askopenfilename(
    title="Open a CSV dataset",
    filetypes=[("CSV files", "*.csv"), ("All files", "*.*")],
)
if selected:
    path = Path(selected)
    try:
        frame = pd.read_csv(path)
        print(f"Loaded {len(frame):,} rows from {path.name}")
    except (OSError, UnicodeError, pd.errors.ParserError) as exc:
        messagebox.showerror("Could not open file", str(exc), parent=root)
else:
    print("Open cancelled")
root.destroy()

## 3. Paths, encoding, and overwrite policy

Use `pathlib.Path` after the dialog returns a path to inspect extensions, compose output locations, and create directories. Do not assume that a path is writable simply because the user selected it. Opening a file can fail due to permissions, locking, a missing drive, or an invalid workbook. Catch expected exceptions around the specific I/O operation and preserve the error details for logs.

A save workflow should define overwrite behavior. Ask for confirmation before replacing an existing report, write to a temporary path when partial outputs would be dangerous, and only report success after the write completes. For Excel workflows, an unsuccessful export should not leave the user thinking a previous file was updated.

## Guided lab: Lab: open and export workflow

Add Open CSV, Open Excel, Export CSV, Export Excel, and Exit commands. Handle cancellation, unreadable files, unsupported extensions, overwrite confirmation, and write failures. Use the same service methods for menu and toolbar actions. The UI must only display “Saved successfully” after the I/O operation has actually succeeded.

## Review questions and practice

1. What value does a file-open dialog return when the user cancels?
2. Why is choosing a save path not the same as writing a file?
3. How should a Yes/No confirmation affect program flow?
4. Name three failures an export workflow should handle.

## Further reading

- File dialogs: https://docs.python.org/3/library/dialog.html#module-tkinter.filedialog
- Message boxes: https://docs.python.org/3/library/tkinter.messagebox.html

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.